# 08 – Finding 6: what happens with impossible inputs?
**Test:** take the applicants the model approves and set one field to a value that can't be real.
**Expected:** an error, or at least no approval.

In [1]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from common import load_data, predict, what_if

df = load_data()
_, rejected = predict(df)
approved = df[~rejected]
print("Model-approved applicants used:", len(approved))

Model-approved applicants used: 245


In [2]:
cases = {
    "monthly_income = 0": {"monthly_income": 0},
    "loan_amount = 0": {"loan_amount": 0},
    "loan_amount = -1,00,000": {"loan_amount": -100000},
    "loan_amount = 1 crore": {"loan_amount": 10000000},
    "tenure_months = 13 (not offered)": {"tenure_months": 13},
    "age = 5": {"age": 5},
    "age = 120": {"age": 120},
    "credit_score = 2000": {"credit_score": 2000},
    "pincode = 9999999": {"pincode": 9999999},
}
rows = []
for name, change in cases.items():
    _, r = what_if(approved, **change)
    rows.append({"input": name, "still_approved": round(1 - r.mean(), 3)})
table = pd.DataFrame(rows)
table

,input,still_approved
0,monthly_income = 0,0.233
1,loan_amount = 0,1.000
2,"loan_amount = -1,00,000",1.000
3,loan_amount = 1 crore,0.000
4,tenure_months = 13 (not offered),0.522
5,age = 5,0.024
6,age = 120,0.731
7,credit_score = 2000,0.000
8,pincode = 9999999,0.000


No error is raised for any of these. **Zero income still gets 23% approved**, and a negative loan
gets 100% approved.

## Check: do tiny, meaningless changes flip decisions?
400 random applicants (fixed seed), each nudged by a trivial amount.

In [3]:
sample = df.sample(400, random_state=42)
_, before = predict(sample)
small = {
    "income + Rs1": sample.assign(monthly_income=sample.monthly_income + 1),
    "credit score + 1": sample.assign(credit_score=sample.credit_score + 1),
    "loan + Rs5,000": sample.assign(loan_amount=sample.loan_amount + 5000),
    "pincode + 1": sample.assign(pincode=sample.pincode + 1),
}
rows = []
for name, changed in small.items():
    _, after = predict(changed)
    rows.append({"change": name, "decisions_flipped": round((after != before).mean(), 4)})
table = pd.DataFrame(rows)
table

,change,decisions_flipped
0,income + Rs1,0.0000
1,credit score + 1,0.0050
2,"loan + Rs5,000",0.0050
3,pincode + 1,0.0025


**Conclusion:** the model is stable for small changes (this one turned out fine), but it accepts
impossible values and gives confident decisions for them.